# Regression Curve

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.colors as mcolors

trial = 2
ptime_label = "normal"
dataset = "train"  # or "train"

# Load data and filter branches
df = pd.read_csv(f'./trial{trial}_{dataset}_unpre.csv')
df = df[df['branch'].isin([f'{ptime_label}_branch', 'common_branch'])]
df = df.sort_values(f'ptime_{ptime_label}')
ptime = df[f'ptime_{ptime_label}'].values.reshape(-1, 1)  # (N, 1)

# Find the ptime value of split branch
df_common = df[df["branch"] == "common_branch"]
last_cluster = df_common["cluster"].max()
ptime_split = df_common.loc[df_common["cluster"] == last_cluster, f"ptime_{ptime_label}"].mean()

# Extract all feature columns
feature_cols = [col for col in df.columns if col.startswith('CTX_')]

# Create a grid for the x-axis to plot the smoothed curves
ptime_grid = np.linspace(ptime.min(), ptime.max(), 200).reshape(-1, 1)

# Create the plot
fig, ax = plt.subplots(figsize=(3, 2), dpi=200)

# Get colors for the lines (using matplotlib's tab10 colormap)
colors = plt.cm.tab10(np.linspace(0, 1, min(len(feature_cols), 10)))
if len(feature_cols) > 10:
    # If more than 10 features, cycle through colors
    colors = [plt.cm.tab10(i % 10) for i in range(len(feature_cols))]

for i, feature in enumerate(feature_cols):
    y = df[feature].values.reshape(-1, 1)
    
    # Polynomial regression
    poly = PolynomialFeatures(degree=8)
    X_poly = poly.fit_transform(ptime)
    model = LinearRegression().fit(X_poly, y)
    
    # Predict
    y_pred = model.predict(poly.transform(ptime_grid)).flatten()

    # Plot the line
    ax.plot(ptime_grid.flatten(), y_pred, 
            color=colors[i % len(colors)], 
            linewidth=0.8, 
            alpha=0.6, 
            label=feature)

# Add vertical line for split point
ax.axvline(x=ptime_split, color='red', linewidth=2, linestyle='--', 
           label=f'Split at {ptime_split:.3f}')

# Set labels and title
ax.set_xlabel(f'Pseudotime ({dataset})')
ax.set_ylabel('Feature Value')
ax.set_xlim(0, ptime.max())
ax.set_ylim(0.5, 2.5)
# ax.set_title(f'Feature Regression Curves Over Pseudotime ({ptime_label} + Common) - {dataset}')

handles, labels = ax.get_legend_handles_labels()
ax.legend([handles[-1]], [labels[-1]], loc='upper left')

# Add grid for better readability
ax.grid(True, alpha=0.3)

# Adjust layout to prevent legend cutoff
plt.tight_layout()

# Show the plot
plt.show()

# Other data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.colors as mcolors
import random
from scipy.stats import pearsonr

def calculate_pearson_correlation(df, col1, col2):
    r, p = pearsonr(df[col1], df[col2])
    return r, p

def get_pearson_text(r, p):
    pearson_text = ''
    if abs(r) < 0.0001:
        pearson_text += 'r<0.0001, '
    else:
        pearson_text += f'r={r:.4f}, '
    if abs(p) < 0.0001:
        pearson_text += 'p<0.0001'
    else:
        pearson_text += f'p={p:.4f}'
    return pearson_text


# ptime_label = "AD"
# dataset = "train"  # or "train"
trial = 2
prefix = "CTX_"
y_label_dict = {
    "CTX_": "SUVR Value",
    "PHC_MEM": "Memory Score",
    "PHC_EXF": "Executive Function",
    "age": "Age"
}

colors = None

for ptime_label in ["AD", "normal"]:
    for dataset in ["train", "test", "single", "NACC_single"]:
        # Load data and filter branches
        df = pd.read_csv(f'./trial{trial}_{dataset}_pre.csv')  # RID, age, CTX_...

        df = df[df['branch'].isin([f'{ptime_label}_branch', 'common_branch'])]
        df = df.sort_values(f'ptime_{ptime_label}')
        ptime = df[f'ptime_{ptime_label}'].values.reshape(-1, 1)  # (N, 1)

        # Find the ptime value of split branch
        df_common = df[df["branch"] == "common_branch"]
        last_cluster = df_common["cluster"].max()
        if dataset == "train":
            ptime_split = df_common.loc[df_common["cluster"] == last_cluster, f"ptime_{ptime_label}"].mean()

        # Extract all feature columns
        feature_cols = [col for col in df.columns if col.startswith(f'{prefix}')]
        # Get colors for the lines (using matplotlib's tab10 colormap)
        if colors is None:
            colors = plt.cm.tab10(np.linspace(0, 1, 10))
            if len(feature_cols) > 10:
                # If more than 10 features, cycle through colors
                colors = [plt.cm.tab10(i % 10) for i in range(len(feature_cols))]
            elif len(feature_cols) == 1:
                colors = [colors[random.randint(0, 9)]]

        # Create a grid for the x-axis to plot the smoothed curves
        ptime_grid = np.linspace(ptime.min(), ptime.max(), 200).reshape(-1, 1)

        # Create the plot
        fig, ax = plt.subplots(figsize=(3, 2.25), dpi=200)

        for i, feature in enumerate(feature_cols):
            y = df[feature].values.reshape(-1, 1)
            X = ptime
            # filter out NaN values
            mask = ~np.isnan(y).flatten()
            y = y[mask]
            X = X[mask]

            if len(feature_cols) == 1:
                ax.scatter(X.flatten(), y.flatten(), 
                          color=colors[0], s=4, alpha=0.1)
                # Calculate and display Pearson correlation
                r, p = calculate_pearson_correlation(df, f'ptime_{ptime_label}', feature)
                pearson_text = get_pearson_text(r, p)
                ax.text(0.05, 0.95, pearson_text, transform=ax.transAxes, fontsize=8, verticalalignment='top')

            # Polynomial regression
            poly = PolynomialFeatures(degree=8 if len(feature_cols) > 1 else 1)
            X_poly = poly.fit_transform(X)
            model = LinearRegression().fit(X_poly, y)
            
            # Predict
            y_pred = model.predict(poly.transform(ptime_grid)).flatten()

            # Plot the line
            linewidth = 0.8 if prefix == "CTX_" else 2
            line_color = colors[i % len(colors)]
            ax.plot(ptime_grid.flatten(), y_pred, 
                    color=line_color, 
                    linewidth=linewidth, 
                    alpha=0.6, 
                    label=feature)
            
            
        # Add vertical line for split point
        ax.axvline(x=ptime_split, color='red', linewidth=1.5, linestyle='--', 
                label=f'Split at {ptime_split:.3f}')

        # Set labels and title
        ax.set_xlabel(f'Pseudotime ({dataset})')
        ax.set_ylabel(y_label_dict[prefix])
        ax.set_xlim(0, 1)
        if prefix == "CTX_":
            ax.set_ylim(0.2, 2.5)
        elif prefix == "PHC_MEM":
            ax.set_ylim(-2, 3)
        elif prefix == "PHC_EXF":
            ax.set_ylim(-1.8, 1.8)
        elif prefix == "age":
            ax.set_ylim(55, 99)
        # ax.set_title(f'Feature Regression Curves Over Pseudotime ({ptime_label} + Common) - {dataset}')

        handles, labels = ax.get_legend_handles_labels()
        loc = 'upper left' if prefix == "CTX_" else 'upper right'
        # ax.legend([handles[-1]], [labels[-1]], loc=loc)

        # Add grid for better readability
        ax.set_xticks(np.arange(0, 1.1, 0.2))
        ax.grid(True, alpha=0.3)

        # Adjust layout to prevent legend cutoff
        plt.tight_layout()

        # Show the plot
        plt.show()

---